# 06 - FedAvg vs FedProx + biaya komunikasi (Paper 4, Fase 2) - H2/efisiensi

**Tujuan:** (1) bandingkan **FedAvg** vs **FedProx** (proximal `mu`) di bawah
non-IID; FedProx seharusnya lebih stabil saat klien menyimpang. (2) laporkan
**biaya komunikasi**: rounds-to-converge & total bytes bobot dikirim vs MCC
(kurva biaya-komunikasi).

**Konsistensi:** `../aws/fed_node.py` SUDAH mendukung FedProx (`mu>0` di
`train_local`). Notebook ini memakai fungsi yang SAMA -> hasil = prediksi EC2.

Output: `fed_out/fedprox_comm.csv` (metrik + biaya per konfigurasi) +
`fed_out/fedprox_curve.csv` (MCC vs round untuk tiap mu).

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + impor fed_node

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, f1_score, balanced_accuracy_score
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=50; LOCAL_EPOCHS=1; LR=1e-3; SEED=42
print('fed_node loaded | LAYERS', fed.LAYERS)

## 1. Muat ruang SFM + partisi non-IID (reuse pola nb05)

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
def dirichlet_partition(X, y, K, alpha, seed=SEED):
    r=np.random.RandomState(seed); idx_by_c={c:np.where(y==c)[0] for c in np.unique(y)}
    for c in idx_by_c: r.shuffle(idx_by_c[c])
    ci=[[] for _ in range(K)]
    for c,idc in idx_by_c.items():
        prop=r.dirichlet([alpha]*K); cuts=(np.cumsum(prop)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc,cuts)): ci[k].extend(part.tolist())
    parts={}
    for k in range(K):
        ii=np.array(ci[k],int); r.shuffle(ii)
        if len(ii): parts[f'c{k}']=(X[ii],y[ii])
    return parts
if ok:
    Xtr=np.vstack([Xc_tr,Xu_tr]); ytr=np.concatenate([yc_tr,yu_tr])
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    parts=dirichlet_partition(Xtr,ytr,K=6,alpha=0.1,seed=SEED)   # non-IID menantang
    print('klien non-IID:', len(parts))
else:
    print('(partisi SFM belum ada - jalankan nb01)')

## 2. FedAvg vs FedProx + pencatatan biaya komunikasi

Biaya komunikasi: tiap round, tiap klien mengirim seluruh bobot (float64).
`bytes/round = n_clients * total_params * 8`. `rounds_to_converge` = round pertama
saat MCC mencapai >=99% MCC akhir (ambang sederhana, dilaporkan apa adanya).

In [ ]:
def total_params(W): return int(sum(w.size for w in W))
def run(mu, parts, rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    zc={cid:fed.zscore_fit(X) for cid,(X,y) in parts.items()}
    Xz={cid:fed.zscore_apply(parts[cid][0],*zc[cid]) for cid in parts}
    sizes={cid:len(parts[cid][1]) for cid in parts}
    Wg=fed.init_weights(seed); mu_g,sd_g=fed.zscore_fit(np.vstack([p[0] for p in parts.values()]))
    curve=[]
    for t in range(rounds):
        cw=[];cs=[]
        for cid in parts:
            Wl=fed.train_local(Wg,Xz[cid],parts[cid][1],epochs=epochs,lr=lr,mu=mu,W_global=Wg,seed=seed)
            cw.append(Wl);cs.append(sizes[cid])
        Wg=fed.fedavg(cw,cs)
        out,_=fed.forward(Wg,fed.zscore_apply(Xg_te,mu_g,sd_g))
        curve.append(round(matthews_corrcoef(yg_te,(out>=0.5).astype(int)),4))
    P=total_params(Wg); bpr=len(parts)*P*8
    final=curve[-1]
    rtc=next((i for i,v in enumerate(curve) if final>0 and v>=0.99*final), rounds)
    return Wg, curve, dict(params=P, bytes_per_round=bpr, rounds_to_converge=rtc,
                           total_bytes=bpr*(rtc+1), MCC_final=final)
res=[]; curves={}
if ok:
    for mu in (0.0, 0.001, 0.01, 0.1):
        _,curve,info=run(mu,parts)
        name='FedAvg' if mu==0.0 else f'FedProx(mu={mu})'
        res.append({'method':name,'mu':mu, **info}); curves[name]=curve
        print(f'{name}: MCC={info["MCC_final"]} rtc={info["rounds_to_converge"]} bytes/round={info["bytes_per_round"]}')
    print('selesai')
else:
    print('(lewati)')
print('=== SEL 2 (FedAvg vs FedProx + biaya) SELESAI ===')

## 3. Simpan hasil + kurva (tabel G)

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'fedprox_comm.csv'),index=False)
    cur=pd.DataFrame({'round':list(range(ROUNDS))}); 
    for k,v in curves.items(): cur[k]=v
    cur.to_csv(os.path.join(OUTDIR,'fedprox_curve.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    print('H2/efisiensi: bandingkan MCC_final & rounds_to_converge FedAvg vs FedProx di non-IID.')
else:
    print('(tak ada hasil)')
print('=== SEL 3 (simpan) SELESAI ===')

## 4. Catatan untuk naskah

- Hasil -> **Tabel G** (`documentation_federated.md`): FedAvg vs FedProx + biaya.
- Kurva `fedprox_curve.csv` -> Gambar MCC-vs-round per mu (`figure-fed/`).
- Biaya komunikasi dilaporkan transparan (params, bytes/round, rounds-to-converge);
  ambang konvergensi 99% MCC akhir dinyatakan eksplisit (bukan angka ajaib).
- FedProx `mu` dipilih via hasil ini; dipakai sbg konfigurasi final bila lebih baik.
- Logika identik `aws/fed_node.py` -> validasi skrip EC2 sebelum deploy.